<a href="https://colab.research.google.com/github/colab-patricio/studiesIn-systemsAnalysis-Development/blob/main/Gradua%C3%A7%C3%A3o/1%C2%BAperiodo/Arq.%20e%20organiza%C3%A7%C3%A3o%20de%20computadores/AF_arqOrganizacaoComputadores_edge_computing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

a81fe062-7044-4dde-96fb93c017e57217.avif

# Edge Computing e Arquiteturas Distribuídas

**Objetivo deste notebook:** complementar a apresentação teórica com uma análise quantitativa baseada em dados públicos reais sobre por que a computação de borda (edge computing) se tornou necessária em três contextos: veículos autônomos (Tesla e Waymo) e, de forma mais qualitativa, a Indústria 4.0.

**Estrutura:**
1. Evolução do hardware de borda da Tesla (FSD Computer)
2. Por que enviar tudo para a nuvem é inviável: banda e custo (case Waymo)
3. Latência: processamento local vs. round-trip de rede
4. Funil de redução de dados via pré-processamento na borda
5. Síntese comparativa dos três cases

> **Nota sobre metodologia:** todos os números marcados como *"confirmado"* vêm diretamente de fontes técnicas ou de imprensa especializada citadas em cada seção. Números marcados como *"estimado"* ou *"exercício didático"* são cálculos feitos a partir desses dados públicos para fins de ilustração — e não valores oficiais divulgados pelas empresas.

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
print("Bibliotecas carregadas.")

## 1. Evolução do FSD Computer da Tesla

A Tesla é o exemplo mais direto de edge computing em veículos: em vez de depender de conexão com a nuvem, todo o processamento de percepção e decisão acontece dentro do carro, em tempo real.

- **HW3 (2019):** dois chips redundantes de 72 TOPS cada, totalizando **144 TOPS**; processa até **1 bilhão de pixels/segundo** vindos de 8 câmeras.
- **HW4 (2023):** throughput de inferência de aproximadamente **1,3 gigapixels/segundo**, descrito por fontes especializadas como um ganho de "3 a 5 vezes" em relação ao HW3. **Importante:** não encontramos um número oficial de TOPS para o HW4 consistente entre fontes confiáveis — por isso tratamos esse valor como uma **estimativa em faixa**, não um número exato.
- **HW4.1 (anunciado em 2026):** +10% de poder de computação e +10% de banda de memória sobre o HW4; RAM total dobra de 32 GB (2×16 GB) para 64 GB (2×32 GB).

O gráfico abaixo usa barras de erro para deixar essa incerteza visível — uma escolha deliberada para a apresentação: mostrar que nem todo dado técnico de empresas privadas é público com precisão, e que isso deve ser comunicado, não escondido.

In [ ]:
geracoes = ["HW3\n(2019)", "HW4\n(2023)", "HW4.1\n(anunciado 2026)"]

# TOPS confirmado apenas para o HW3. Para HW4/HW4.1 usamos a faixa de 3x-5x
# mencionada por fontes de imprensa especializada (ponto médio = 4x) e sinalizamos
# a incerteza com barras de erro.
tops_medio = [144, 144*4, 144*4*1.10]
erro_inferior = [0, 144*4 - 144*3, (144*4*1.10) - (144*3*1.10)]
erro_superior = [0, 144*5 - 144*4, (144*5*1.10) - (144*4*1.10)]

fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(geracoes))
cores = ["#2c7fb8", "#7fcdbb", "#c7e9b4"]
ax.bar(x, tops_medio, color=cores, width=0.55)
ax.errorbar(x, tops_medio, yerr=[erro_inferior, erro_superior],
            fmt="none", ecolor="black", capsize=6)

ax.set_xticks(x)
ax.set_xticklabels(geracoes)
ax.set_ylabel("TOPS (trilhões de operações/s)")
ax.set_title("Evolução do FSD Computer da Tesla\n(HW3 confirmado; HW4/HW4.1 estimados — ver texto acima)")
ax.text(0, 144 + 15, "144\n(confirmado)", ha="center", fontsize=9, fontweight="bold")
ax.text(1, 144*4 + 40, "~432–720\n(estimado, 3x–5x)", ha="center", fontsize=9)
ax.text(2, 144*4*1.10 + 40, "~475–790\n(estimado, +10%)", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Por que não simplesmente enviar tudo para a nuvem? (case Waymo)

Um relatório de infraestrutura de IA para veículos autônomos (Introl, 2026) traz um dado central para este argumento: a frota da Waymo gera **25 TB de dados de sensores por veículo, por dia**, exigindo processamento local equivalente a **200 TFLOPS** com latência abaixo de **10 ms** para decisões críticas de segurança. O mesmo relatório indica que o **pré-processamento na borda reduz o volume bruto em 10x** antes da inferência da rede neural.

Vamos transformar isso em dois números concretos: (1) a banda de rede que seria necessária para transmitir tudo em tempo real, e (2) uma estimativa de custo de transferência de dados através de um **exercício didático nosso**, não um valor divulgado pela Waymo.

In [ ]:
TB_por_dia = 25  # dado da fonte (Introl, 2025)
bytes_por_dia = TB_por_dia * (10**12)
segundos_por_dia = 24 * 3600
banda_media_mbps = (bytes_por_dia * 8) / segundos_por_dia / 1e6

reducao_edge = 10  # fator de redução reportado pela fonte
volume_pos_edge_TB = TB_por_dia / reducao_edge
banda_pos_edge_mbps = banda_media_mbps / reducao_edge

print(f"Volume bruto gerado por veículo:           {TB_por_dia} TB/dia")
print(f"Banda MÉDIA sustentada p/ enviar tudo cru:  {banda_media_mbps:,.0f} Mbps (~{banda_media_mbps/1000:.2f} Gbps)")
print(f"Volume após pré-processamento na borda:     {volume_pos_edge_TB:.1f} TB/dia")
print(f"Banda necessária após o pré-processamento:  {banda_pos_edge_mbps:,.0f} Mbps")
print()
print("Para efeito de comparação: uma conexão 4G/LTE tipica sustenta dezenas de Mbps;")
print("uma boa conexão 5G, algumas centenas. Enviar ~2,3 Gbps de forma continua e")
print("simultanea por uma frota inteira de veiculos e, na pratica, inviavel — daí a")
print("necessidade do processamento local.")

In [ ]:
# --- EXERCÍCIO DIDÁTICO: custo hipotético de transferência de dados ---
# Não são valores reais divulgados pela Waymo. Usamos uma faixa de preços públicos
# tipicamente cobrados por provedores de nuvem para transferência de dados (egress),
# apenas para ilustrar a ORDEM DE GRANDEZA do problema.
preco_min_usd_por_GB = 0.05
preco_max_usd_por_GB = 0.09
GB_por_dia = TB_por_dia * 1000

custo_bruto = (GB_por_dia * preco_min_usd_por_GB, GB_por_dia * preco_max_usd_por_GB)
custo_com_edge = ((GB_por_dia/reducao_edge) * preco_min_usd_por_GB,
                   (GB_por_dia/reducao_edge) * preco_max_usd_por_GB)

print(f"Custo de banda SEM pré-processamento na borda: US$ {custo_bruto[0]:,.0f} a US$ {custo_bruto[1]:,.0f} / veículo / dia")
print(f"Custo de banda COM pré-processamento na borda: US$ {custo_com_edge[0]:,.0f} a US$ {custo_com_edge[1]:,.0f} / veículo / dia")

frota_hipotetica = 1000  # número redondo, apenas para ilustrar escala — não é o tamanho real de uma frota específica
print(f"\nEm escala de uma frota hipotética de {frota_hipotetica} veículos (apenas para ilustrar ordem de grandeza):")
print(f"  Sem edge:  US$ {custo_bruto[0]*frota_hipotetica:,.0f} a US$ {custo_bruto[1]*frota_hipotetica:,.0f} / dia")
print(f"  Com edge:  US$ {custo_com_edge[0]*frota_hipotetica:,.0f} a US$ {custo_com_edge[1]*frota_hipotetica:,.0f} / dia")

## 3. Latência: processamento local vs. round-trip de rede

Além do custo e da banda, existe um limite físico: **decisões de segurança crítica não podem esperar uma viagem de ida e volta até um servidor remoto.**

- A pilha de percepção da Waymo, rodando na borda, alcança cerca de **3 ms de latência ponta a ponta** (fonte citada acima).
- A literatura de redes móveis documenta faixas típicas de latência de ida-e-volta (RTT): 5G em boas condições fica entre ~10-20 ms; 4G/LTE, entre ~40-60 ms. Esses valores são ordens de grandeza típicas da tecnologia, não medições de um veículo específico.
- Um limite de referência comumente citado na indústria automotiva para decisões críticas (frenagem de emergência, desvio) é de aproximadamente **10 ms**.

O gráfico deixa claro por que a arquitetura de borda não é uma escolha de otimização, mas uma exigência física do problema.

In [ ]:
cenarios = ["Processamento na borda\n(Waymo, medido)", "Rede 5G\n(condição boa, típico)", "Rede 4G/LTE\n(típico)"]
lat_min = [3, 10, 40]
lat_max = [3, 20, 60]
limite_seguranca = 10

medios = [(a+b)/2 for a, b in zip(lat_min, lat_max)]
erros = [(b-a)/2 for a, b in zip(lat_min, lat_max)]
cores = ["#2ca25f" if m <= limite_seguranca else "#de2d26" for m in medios]

fig, ax = plt.subplots(figsize=(9, 5.5))
x = np.arange(len(cenarios))
ax.bar(x, medios, yerr=erros, capsize=6, color=cores)
ax.axhline(limite_seguranca, color="black", linestyle="--", linewidth=1.2)
ax.text(2.05, limite_seguranca + 1, "limite crítico de segurança (~10 ms)",
        fontsize=9, ha="right", va="bottom")
ax.set_xticks(x)
ax.set_xticklabels(cenarios)
ax.set_ylabel("Latência (ms)")
ax.set_title("Por que decisões críticas não podem esperar a nuvem")
plt.tight_layout()
plt.show()

## 4. Funil de redução de dados via pré-processamento na borda

Juntando os números da Seção 2: o volume de dados gerado por sensor é muito maior do que o volume que, de fato, precisa sair do veículo. O pré-processamento na borda funciona como um filtro — só o que é relevante (eventos, metadados, trechos selecionados) segue para a nuvem, geralmente em lote, quando o veículo está em uma rede de melhor qualidade (ex. Wi-Fi na garagem).

In [ ]:
etapas = ["Dados brutos\ngerados pelos sensores", "Após pré-processamento\nna borda (edge)", "Enviado à nuvem\n(treinamento/analytics, em lote)"]
volumes_TB = [25, 2.5, 2.5]

fig, ax = plt.subplots(figsize=(8, 5))
cores = ["#08519c", "#6baed6", "#c6dbef"]
bars = ax.bar(etapas, volumes_TB, color=cores, width=0.6)
for b, v in zip(bars, volumes_TB):
    ax.text(b.get_x() + b.get_width()/2, v + 0.5, f"{v} TB/dia", ha="center", fontweight="bold")
ax.set_ylabel("Volume de dados (TB/dia, por veículo)")
ax.set_title("Redução de volume de dados via pré-processamento na borda")
plt.tight_layout()
plt.show()

## 5. Síntese comparativa: três cases, um mesmo princípio

| Case | Volume/exigência de dados | Restrição de latência | Estratégia de borda |
|---|---|---|---|
| **Tesla (FSD)** | ~1–1,3 gigapixels/s de vídeo de 8 câmeras, processados continuamente | Decisão de direção em tempo real; não tolera round-trip de rede | Todo o processamento de percepção e decisão ocorre em chips proprietários dentro do veículo (144 TOPS confirmados no HW3) |
| **Waymo** | 25 TB/dia por veículo | <10 ms para decisões críticas | ~200 TFLOPS de processamento local; pré-processamento reduz 10x o volume antes de qualquer envio à nuvem |
| **Indústria 4.0 (ex. Siemens Industrial Edge)** | Telemetria contínua de sensores de chão de fábrica | Milissegundos para parar uma máquina antes de uma falha ou desperdício de material | Processamento local (edge) para decisões operacionais; dados agregados sobem à nuvem para analytics e manutenção preditiva |

**Observação sobre o case de Indústria 4.0:** ao contrário dos dois primeiros, as fontes públicas encontradas sobre a Siemens são majoritariamente material institucional/marketing, sem números tão precisos quanto os do setor automotivo.



### Conclusão
Os três cases compartilham a mesma lógica causal: **quando o custo de banda, a latência de rede ou ambos tornam o processamento em nuvem inviável para uma decisão que precisa ser tomada agora, a inteligência precisa se mover para perto de onde o dado é gerado.** Edge computing não é uma tendência de marketing — é uma resposta a um limite físico (velocidade da luz / latência de rede) e a um limite econômico (custo de banda em escala).

### Fontes utilizadas
- [Tesla FSD Hardware 3 vs Hardware 4: Technical Deep Dive](https://www.linkedin.com/pulse/tesla-fsd-hardware-3-vs-4-technical-deep-dive-vikas-pandit-k23nc/)
- [Autonomous Vehicle AI Infrastructure: Edge-to-Cloud GPU Requirements](https://introl.com/blog/autonomous-vehicle-ai-infrastructure-edge-cloud)